# 🧠 VedaAI — Complete Algorithm Documentation

This notebook documents every algorithm used in the VedaAI assessment platform:

1. **Text Extraction** — PyMuPDF, Tesseract OCR, Gemini Vision OCR
2. **Question Parsing** — Regex patterns for Q/प्रश्न/Question extraction
3. **Answer Segment Detection** — Marker-based word segmentation
4. **Answer Matching** — Number matching + fuzzy text similarity
5. **OCR Highlighting** — Bounding box calculation for answer regions
6. **Multi-page Answer Merging** — Cross-page answer detection
7. **AI Evaluation** — Subject-aware grading with Gemini + local fallback
8. **Deduplication** — Duplicate segment removal

## 0. Setup & Installation

In [ ]:
# Install required packages
!pip install PyMuPDF pytesseract pdf2image rapidfuzz Pillow google-generativeai 2>/dev/null | tail -5
!apt-get install -y tesseract-ocr tesseract-ocr-hin 2>/dev/null | tail -3

In [ ]:
import fitz  # PyMuPDF
import re
import os
import json
import base64
import mimetypes
from PIL import Image
from rapidfuzz import fuzz

print("✅ All imports successful")
print(f"PyMuPDF version: {fitz.__version__}")

---
## 1. Text Extraction Algorithm

VedaAI uses **three different OCR engines** depending on the input file type:

| Engine | Input | How it works |
|--------|-------|-------------|
| **PyMuPDF** | PDF | Direct text layer extraction (fastest, best for digital PDFs) |
| **Tesseract** | PDF/Image | Traditional OCR with Hindi+English support |
| **Gemini Vision** | Image | AI-powered OCR with layout understanding |

### 1a. PyMuPDF Text Extraction (PDF → Words with coordinates)

PyMuPDF extracts the **embedded text layer** from PDFs. Each word comes with:
- `text`: The word content
- `x0, y0, x1, y1`: Bounding box coordinates (in points)
- `block, line, word`: Structural indices

This is the **fastest** method and gives perfect results for digitally-created PDFs.

In [ ]:
def extract_words_pymupdf(pdf_path):
    """
    ALGORITHM: PyMuPDF Word Extraction
    
    1. Open PDF document with fitz.open()
    2. For each page, call page.get_text('words')
    3. Each word tuple: (x0, y0, x1, y1, text, block_no, line_no, word_no)
    4. Convert to structured dict with coordinates
    
    Output: List of pages, each containing words with pixel coordinates
    """
    doc = fitz.open(pdf_path)
    pages = []
    
    for page_number, page in enumerate(doc, start=1):
        words = page.get_text("words")  # Returns list of tuples
        
        page_words = []
        for word in words:
            x0, y0, x1, y1, text, block, line, word_no = word
            page_words.append({
                "text": text,
                "x0": x0, "y0": y0,  # Top-left corner
                "x1": x1, "y1": y1,  # Bottom-right corner
                "block": block,
                "line": line,
                "word": word_no
            })
        
        pages.append({
            "page": page_number,
            "words": page_words,
            "width": page.rect.width,
            "height": page.rect.height
        })
    
    doc.close()
    return pages

# Demo with a sample
# pages = extract_words_pymupdf('sample.pdf')
# print(f"Page 1: {len(pages[0]['words'])} words found")
# for w in pages[0]['words'][:5]:
#     print(f"  '{w['text']}' at ({w['x0']:.0f},{w['y0']:.0f})-({w['x1']:.0f},{w['y1']:.0f})")

### 1b. Tesseract OCR (Scanned PDFs → Words)

For scanned PDFs where PyMuPDF can't extract text, we fall back to Tesseract:

1. Convert PDF page to image at 300 DPI
2. Run Tesseract OCR with `--psm 6` (uniform block of text)
3. Filter words with confidence < 30%
4. Return words with bounding boxes

In [ ]:
try:
    import pytesseract
    from pdf2image import convert_from_path
    TESSERACT_AVAILABLE = True
except ImportError:
    TESSERACT_AVAILABLE = False
    print("⚠️ Tesseract not available — install with: apt install tesseract-ocr")

def extract_words_tesseract(pdf_path, dpi=300):
    """
    ALGORITHM: Tesseract OCR Extraction
    
    1. Convert PDF pages to images at 300 DPI
    2. Run pytesseract.image_to_data() with --psm 6
       (Page Segmentation Mode 6 = assume uniform text block)
    3. Filter: confidence >= 30, non-empty text
    4. Extract bounding boxes from OCR data
    
    Auto-fallback: If PyMuPDF extracts < 100 chars total,
    Tesseract runs as backup.
    """
    if not TESSERACT_AVAILABLE:
        return []
    
    pages = []
    images = convert_from_path(pdf_path, dpi=dpi)
    
    for page_number, image in enumerate(images, start=1):
        img_width, img_height = image.size
        
        # Tesseract OCR with Hindi+English
        ocr_data = pytesseract.image_to_data(
            image,
            output_type=pytesseract.Output.DICT,
            config="--psm 6"
        )
        
        page_words = []
        n_boxes = len(ocr_data["text"])
        
        for i in range(n_boxes):
            text = ocr_data["text"][i].strip()
            if not text:
                continue
            
            conf = int(ocr_data["conf"][i]) if ocr_data["conf"][i] != "-1" else 0
            if conf < 30:  # Filter low-confidence words
                continue
            
            x = ocr_data["left"][i]
            y = ocr_data["top"][i]
            w = ocr_data["width"][i]
            h = ocr_data["height"][i]
            
            page_words.append({
                "text": text,
                "x0": x, "y0": y,
                "x1": x + w, "y1": y + h,
                "block": ocr_data["block_num"][i],
                "line": ocr_data["line_num"][i],
                "word": ocr_data["word_num"][i],
                "confidence": conf
            })
        
        pages.append({
            "page": page_number,
            "words": page_words,
            "width": img_width,
            "height": img_height
        })
    
    return pages

print("✅ Tesseract extraction function defined")

### 1c. Gemini Vision OCR (Image → Text)

For image files (PNG, JPG, WEBP), we use **Gemini Vision API**:

1. Read image as binary, encode to base64
2. Send to Gemini with prompt: "Extract ALL text as JSON array"
3. Parse response: each word gets `{text, x0, y0, x1, y1}`
4. If JSON fails, treat entire response as single text block

**Why Gemini?** It understands Hindi + English mixed text, tables, handwritten text, and complex layouts that Tesseract struggles with.

In [ ]:
def extract_image_text_gemini(image_path, api_key):
    """
    ALGORITHM: Gemini Vision OCR
    
    1. Read image → base64 encode
    2. Send to Gemini with inline_data + extraction prompt
    3. Parse JSON response for word coordinates
    4. Fallback: if no JSON, return whole text as one block
    
    This handles:
    - Mixed Hindi/English text
    - Handwritten text
    - Complex table layouts
    - Low-quality/scanned images
    """
    import google.generativeai as genai
    genai.configure(api_key=api_key)
    model = genai.GenerativeModel('gemini-2.5-flash')
    
    with open(image_path, "rb") as f:
        image_data = f.read()
    
    mime, _ = mimetypes.guess_type(image_path)
    mime = mime or "image/png"
    
    response = model.generate_content([
        {
            "inline_data": {
                "mime_type": mime,
                "data": base64.b64encode(image_data).decode()
            }
        },
        "Extract ALL text from this image. For each word or text fragment, "
        "return a JSON array like: [{\"text\": \"word\", \"x0\": 10, \"y0\": 20, "
        "\"x1\": 50, \"y1\": 35}]. Use pixel coordinates. Return ONLY the JSON."
    ])
    
    text = response.text.strip()
    
    if text.startswith("["):
        words = json.loads(text)
    else:
        words = [{"text": text, "x0": 0, "y0": 0, "x1": 100, "y1": 100}]
    
    img = Image.open(image_path)
    img_w, img_h = img.size
    
    page_words = []
    for w in words:
        page_words.append({
            "text": w.get("text", ""),
            "x0": float(w.get("x0", 0)),
            "y0": float(w.get("y0", 0)),
            "x1": float(w.get("x1", img_w)),
            "y1": float(w.get("y1", img_h)),
            "block": 0, "line": 0, "word": len(page_words),
        })
    
    return [{"page": 1, "words": page_words, "width": img_w, "height": img_h}]

print("✅ Gemini Vision OCR function defined")

### 1d. Smart Auto-Fallback Selection

The `extract_words()` function automatically picks the best OCR engine:

```
Input file
  ├── Image file (.png/.jpg/.webp) → Gemini Vision OCR
  └── PDF file (.pdf)
       ├── Try PyMuPDF first (fast, embedded text)
       ├── Count total characters extracted
       ├── If chars < 100 (scanned PDF) → Tesseract OCR
       └── If Tesseract gives more chars → use Tesseract
           Otherwise → keep PyMuPDF result
```

In [ ]:
def extract_words_smart(file_path, force_ocr=False, api_key=None):
    """
    ALGORITHM: Smart OCR Engine Selection
    
    Decision tree:
    1. If file is image → Gemini Vision
    2. If force_ocr=True → Tesseract
    3. Try PyMuPDF → count extracted characters
    4. If chars < 100 → try Tesseract
    5. Use whichever gives more text
    """
    ext = os.path.splitext(file_path)[1].lower()
    IMAGE_EXT = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tiff"}
    
    # Path 1: Image file → Gemini Vision
    if ext in IMAGE_EXT:
        if api_key:
            return extract_image_text_gemini(file_path, api_key)
        return []
    
    # Path 2: Force OCR → Tesseract
    if force_ocr:
        return extract_words_tesseract(file_path)
    
    # Path 3: Try PyMuPDF first
    pymupdf_pages = extract_words_pymupdf(file_path)
    total_chars = sum(
        len(" ".join(w["text"] for w in p["words"]))
        for p in pymupdf_pages
    )
    
    print(f"  PyMuPDF extracted {total_chars} characters")
    
    # Path 4: If too few chars → try Tesseract
    if total_chars < 100 and TESSERACT_AVAILABLE:
        print("  Few chars detected — trying Tesseract...")
        tesseract_pages = extract_words_tesseract(file_path)
        tesseract_chars = sum(
            len(" ".join(w["text"] for w in p["words"]))
            for p in tesseract_pages
        )
        print(f"  Tesseract extracted {tesseract_chars} characters")
        
        if tesseract_chars > total_chars:
            return tesseract_pages
    
    return pymupdf_pages

print("✅ Smart OCR selection function defined")

---
## 2. Text Normalization

Before any processing, all text goes through normalization:

1. Replace non-breaking spaces (\u00a0) with regular spaces
2. Collapse multiple whitespace into single space
3. Strip leading/trailing whitespace

In [ ]:
def normalize_text(text):
    """
    ALGORITHM: Text Normalization
    
    Input: Raw text from OCR/PDF extraction
    Output: Clean, single-spaced text
    
    Steps:
    1. Replace Unicode non-breaking space (\u00a0) → regular space
    2. Regex: collapse all \s+ sequences → single space
    3. Strip leading/trailing whitespace
    """
    if not text:
        return ""
    text = str(text)
    text = text.replace("\u00a0", " ")       # Non-breaking space → space
    text = re.sub(r"\s+", " ", text)         # Multiple spaces → single
    return text.strip()

# Demo
print(normalize_text("Hello\u00a0world\n\n\tmultiple   spaces"))
# Output: "Hello world multiple spaces"

---
## 3. Question Parsing Algorithm

Extract questions from the question paper text using regex patterns.

**Pattern:** Finds lines starting with Q/प्रश्न/Question followed by a number.

**Sub-questions:** Detects (a), (b), (c) patterns within each question block.

In [ ]:
QUESTION_PATTERN = re.compile(
    r"(?m)^\s*(?:Q|प्रश्न|Question|q)\s*[.:]?\s*(\d+)\s*[.:.\-]\s*"
)

SUBQUESTION_PATTERN = re.compile(
    r"(?m)^\s*\(\s*([a-zA-Z])\s*\)"
)

def extract_questions(file_path):
    """
    ALGORITHM: Question Extraction
    
    Input: Full text of question paper (multi-page)
    Output: List of questions with IDs and text
    
    Steps:
    1. Extract text from all pages (PDF or image)
    2. For each page, find QUESTION_PATTERN matches
    3. For each match, extract question number
    4. Extract text block between this question and next
    5. Remove subject labels (Physics, Chemistry, English)
    6. Check for sub-questions (a), (b), (c)
    7. If sub-questions exist → split into sub-questions
    8. If no sub-questions → treat entire block as one question
    
    Question IDs:
    - Main question: q_21 → display "21"
    - Sub-question: q_21_a → display "21(a)"
    """
    # Get pages based on file type
    ext = os.path.splitext(file_path)[1].lower()
    IMAGE_EXT = {".png", ".jpg", ".jpeg", ".webp"}
    
    if ext in IMAGE_EXT:
        # Gemini OCR for images
        pages = extract_image_text_gemini(file_path, "YOUR_API_KEY")
    else:
        # PyMuPDF for PDFs
        doc = fitz.open(file_path)
        pages = []
        for idx, page in enumerate(doc, start=1):
            pages.append({"page": idx, "text": page.get_text("text")})
        doc.close()
    
    questions = []
    
    for page_data in pages:
        page_number = page_data["page"]
        text = page_data["text"]
        
        # Find all question starts on this page
        matches = list(QUESTION_PATTERN.finditer(text))
        
        for i, match in enumerate(matches):
            number = int(match.group(1))  # Extract question number
            
            # Text block: from end of this match to start of next
            start = match.end()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
            block = text[start:end].strip()
            
            # Remove subject labels
            block = re.sub(r"\b(Physics|Chemistry|English)\b", "", block, flags=re.I)
            block = block.strip()
            
            # Check for sub-questions
            sub_matches = list(SUBQUESTION_PATTERN.finditer(block))
            
            if sub_matches:
                for j, sub in enumerate(sub_matches):
                    letter = sub.group(1).lower()
                    sub_start = sub.end()
                    sub_end = sub_matches[j + 1].start() if j + 1 < len(sub_matches) else len(block)
                    
                    question_text = normalize_text(block[sub_start:sub_end])
                    if question_text:
                        questions.append({
                            "question_id": f"q_{number}_{letter}",
                            "display_number": f"{number}({letter})",
                            "question_text": question_text,
                            "page": page_number
                        })
            else:
                question_text = normalize_text(block)
                if question_text:
                    questions.append({
                        "question_id": f"q_{number}",
                        "display_number": str(number),
                        "question_text": question_text,
                        "page": page_number
                    })
    
    return questions

print("✅ Question extraction function defined")

### 3b. Demo: Question Parsing

In [ ]:
# Simulated question paper text
sample_text = """
प्रश्न 1. भारतीय स्वतंत्रता आंदोलन के बारे में लिखिए।
(a) गांधीजी की भूमिका
(b) नेहरू का योगदान
प्रश्न 2. सिंधु घाटी सभ्यता की विशेषताएँ लिखिए।
प्रश्न 3. रोमन साम्राज्य और गुप्त साम्राज्य की तुलना कीजिए।
"""

matches = list(QUESTION_PATTERN.finditer(sample_text))
print(f"Found {len(matches)} questions:\n")

for m in matches:
    number = m.group(1)
    print(f"  Question {number} found at position {m.start()}-{m.end()}")

# Show sub-question detection
print(f"\nSub-question matches: {len(list(SUBQUESTION_PATTERN.finditer(sample_text)))}")
for sub in SUBQUESTION_PATTERN.finditer(sample_text):
    print(f"  ({sub.group(1)}) at position {sub.start()}")

---
## 4. Answer Segment Detection Algorithm

This is the **core algorithm** — it finds answer boundaries in the answer sheet.

### Multi-pass marker detection:

**PASS 1 — Multi-word patterns** (highest priority):
- `"Answer for Question 5"` / `"for Question 5"` / `"Question 5"`
- Uses sliding window of 3-5 words
- Regex: `(?:answer\s+for\s+|for\s+)?(?:question|q)\s+(\d+)`

**PASS 2 — Single-word patterns** (for unused words only):
- `"5."` / `"Q5"` / `"Ans 5"` / `"प्रश्न 5"`
- Must be standalone (not part of body text)
- Rejects: years (>99), dates, month names

**PASS 3 — Split patterns** (for `Q 5 (a)` across multiple words):
- Detects `"Q"` + `"5"` + `"(a)"` as three separate words
- Merges them into one marker

In [ ]:
ANSWER_MARKER_PATTERN = re.compile(
    r"^\s*(?:Q\s*)?(\d+)\s*(?:[\.\:\-\)]\s*)?(?:\(\s*([a-zA-Z])\s*\))?\s*$",
    re.IGNORECASE,
)

def parse_answer_marker(text):
    """
    ALGORITHM: Answer Marker Detection
    
    Detects if a word is a question marker (e.g., "5.", "Q5", "Ans 5")
    
    Rules:
    1. Must start with Q, Ans, प्रश्न, or be standalone number
    2. Number must be ≤ 99 (reject years like 1947)
    3. Reject if followed by date indicators (ई., AD, BC, months)
    4. Two patterns: with prefix (Q/Ans) or bare number
    
    Returns: (number, letter) or None
    """
    if not text:
        return None
    
    original = str(text).strip()
    
    # Date indicator rejection
    date_indicators = re.compile(
        r"(?:ई\.?|ई\.?पू\.?|AD|BC|CE|BCE|th|st|nd|rd|"
        r"जनवरी|फरवरी|मार्च|अप्रैल|मई|जून|जुलाई|अगस्त|सितंबर|"
        r"अक्टूबर|नवंबर|दिसंबर|"
        r"January|February|March|April|May|June|July|August|"
        r"September|October|November|December|"
        r"स्थापनाएँ|स्थापना|वर्ष|वीटो)",
        re.IGNORECASE,
    )
    if date_indicators.search(original):
        return None
    
    # Pattern 1: With prefix (Q, Ans, प्रश्न)
    pattern_with_prefix = re.compile(
        r"^(?:Q\s*|Ans(?:wer)?\.?\s*|प्रश्न\s*)(\d+)"
        r"\s*(?:\(\s*([a-zA-Z])\s*\)|\s*([a-zA-Z]))?"
        r"\s*[\.\:\-]?\s*$",
        re.IGNORECASE,
    )
    
    # Pattern 2: Bare number (punctuation REQUIRED)
    pattern_bare = re.compile(
        r"^(\d+)\s*(?:\(\s*([a-zA-Z])\s*\)|\s*([a-zA-Z]))?"
        r"\s*[\.\:\-]\s*$",  # '.' is required for bare numbers
        re.IGNORECASE,
    )
    
    match = pattern_with_prefix.match(original)
    if not match:
        match = pattern_bare.match(original)
    
    if not match:
        return None
    
    number = int(match.group(1))
    if number > 99:  # Reject years
        return None
    
    letter = match.group(2) or match.group(3)
    return number, letter.lower() if letter else None

# Demo
test_markers = ["5.", "Q5", "Q 5", "Ans 5", "प्रश्न 5", "1947", "January", "12(st)", "Q (a)"]
for m in test_markers:
    result = parse_answer_marker(m)
    print(f"  '{m}' → {result}")

### 4b. Full Answer Segment Detection

In [ ]:
def find_markers(words):
    """
    ALGORITHM: Multi-Pass Marker Detection
    
    Finds all answer markers on a page using 3 passes:
    
    PASS 1 (Multi-word):
    - Sliding window of 5→3 words
    - Matches: "Answer for Question N" / "for Question N" / "Question N"
    - Marks all indices in window as "used"
    
    PASS 2 (Single-word):
    - Only for words NOT used in Pass 1
    - Matches: "5." / "Q5" / "Ans 5" / "प्रश्न 5"
    
    PASS 3 (Split patterns):
    - Detects "Q" + "5" + "(a)" as separate words
    - Merges into single marker
    
    Deduplication: Keep first occurrence of each (number, letter)
    Sorting: By y-coordinate (top to bottom)
    """
    markers = []
    used_indices = set()
    
    # PASS 1: Multi-word patterns
    multiword_re = re.compile(
        r"(?:answer\s+for\s+|for\s+)?(?:question|q)\s+(\d+)"
        r"(?:\s*\(.*?\))?",
        re.IGNORECASE,
    )
    
    for i in range(len(words)):
        for win in range(5, 2, -1):  # Window 5, 4, 3 words
            if i + win > len(words):
                continue
            phrase = " ".join(str(words[j].get("text", "")).strip() for j in range(i, i + win))
            m = multiword_re.match(phrase)
            if m:
                num = int(m.group(1))
                if num <= 99:
                    for j in range(i, i + win):
                        used_indices.add(j)
                    markers.append({
                        "number": num,
                        "letter": None,
                        "index": i,
                        "y0": float(words[i].get("y0", 0)),
                    })
                    break
    
    # PASS 2: Single-word patterns
    for index, word in enumerate(words):
        if index in used_indices:
            continue
        text = str(word.get("text", "")).strip()
        parsed = parse_answer_marker(text)
        if parsed:
            number, letter = parsed
            used_indices.add(index)
            markers.append({
                "number": number, "letter": letter,
                "index": index,
                "y0": float(word.get("y0", 0)),
            })
    
    # PASS 3: Split patterns (Q + 5 + (a))
    for i in range(len(words) - 2):
        if i in used_indices or (i+1) in used_indices or (i+2) in used_indices:
            continue
        a = str(words[i].get("text", "")).strip()
        b = str(words[i+1].get("text", "")).strip()
        c = str(words[i+2].get("text", "")).strip()
        
        if re.fullmatch(r"[Qq]", a) and re.fullmatch(r"\d+[:.]?", b):
            num_match = re.search(r"\d+", b)
            if num_match and int(num_match.group()) <= 99:
                used_indices.update([i, i+1])
                markers.append({
                    "number": int(num_match.group()),
                    "letter": None,
                    "index": i,
                    "y0": float(words[i].get("y0", 0)),
                })
    
    # Deduplicate: keep first of each (number, letter)
    unique = {}
    for marker in markers:
        key = (marker["number"], marker["letter"])
        if key not in unique:
            unique[key] = marker
    markers = list(unique.values())
    
    # Sort by vertical position (top to bottom)
    markers.sort(key=lambda x: (x["y0"], x["index"]))
    
    return markers

print("✅ Marker detection function defined")

In [ ]:
# Demo: Simulate answer sheet words
sample_words = [
    {"text": "Answer", "x0": 10, "y0": 50, "x1": 60, "y1": 65},
    {"text": "for",    "x0": 65, "y0": 50, "x1": 85, "y1": 65},
    {"text": "Question", "x0": 90, "y0": 50, "x1": 150, "y1": 65},
    {"text": "5.",     "x0": 155, "y0": 50, "x1": 170, "y1": 65},
    {"text": "सिंधु",   "x0": 10, "y0": 70, "x1": 50, "y1": 85},
    {"text": "घाटी",   "x0": 55, "y0": 70, "x1": 90, "y1": 85},
    {"text": "सभ्यता", "x0": 95, "y0": 70, "x1": 150, "y1": 85},
    {"text": "Q",      "x0": 10, "y0": 100, "x1": 20, "y1": 115},
    {"text": "12.",    "x0": 25, "y0": 100, "x1": 45, "y1": 115},
    {"text": "Roman",  "x0": 10, "y0": 120, "x1": 50, "y1": 135},
]

markers = find_markers(sample_words)
print(f"Found {len(markers)} markers:\n")
for m in markers:
    print(f"  Q{m['number']} at word index {m['index']}, y={m['y0']}")

### 4c. Answer Segment Extraction

Once markers are found, each answer segment is extracted as:
- **Start**: Word index of the marker
- **End**: Word index of the next marker (or end of page)
- **Text**: All words between start and end, joined + normalized

In [ ]:
def extract_answer_segments(pdf_path):
    """
    ALGORITHM: Answer Segment Extraction
    
    1. Extract all words with coordinates from answer sheet
    2. Find markers (question numbers) on each page
    3. Split words into segments:
       - Segment N starts at marker N
       - Segment N ends at marker N+1 (or end of page)
    4. Pages WITHOUT markers → marked as "continuation" segments
       (these belong to the previous question's answer)
    
    Output: List of segments with:
    - number: Question number (or None for continuation)
    - page_number, raw_text, words, bounding coords
    - is_continuation: True for markerless pages
    """
    pages_data = extract_words_pymupdf(pdf_path)
    segments = []
    pages_with_markers = {}
    
    for page_data in pages_data:
        page_number = page_data["page"]
        page_width = page_data["width"]
        page_height = page_data["height"]
        words = page_data["words"]
        
        if not words:
            continue
        
        markers = find_markers(words)
        pages_with_markers[page_number] = markers
        
        if not markers:
            continue
        
        # Extract segments between markers
        for i, marker in enumerate(markers):
            start = marker["index"]
            end = markers[i + 1]["index"] if i + 1 < len(markers) else len(words)
            segment_words = words[start:end]
            
            if not segment_words:
                continue
            
            text = " ".join(w.get("text", "") for w in segment_words)
            text = normalize_text(text)
            
            if len(text) < 2:
                continue
            
            segments.append({
                "number": marker["number"],
                "letter": marker["letter"],
                "page_number": page_number,
                "page_width": page_width,
                "page_height": page_height,
                "raw_text": text,
                "words": segment_words,
                "start_index": start,
                "end_index": end,
            })
    
    # Handle pages without markers (continuations)
    all_pages = {p["page"] for p in pages_data}
    for page_num in sorted(all_pages - set(pages_with_markers.keys())):
        page_data = next((p for p in pages_data if p["page"] == page_num), None)
        if not page_data or not page_data["words"]:
            continue
        
        text = " ".join(w.get("text", "") for w in page_data["words"])
        text = normalize_text(text)
        
        if len(text) >= 2:
            segments.append({
                "number": None, "letter": None,
                "page_number": page_num,
                "page_width": page_data["width"],
                "page_height": page_data["height"],
                "raw_text": text,
                "words": page_data["words"],
                "start_index": 0, "end_index": len(page_data["words"]),
                "is_continuation": True,
            })
    
    return segments, pages_with_markers

print("✅ Answer segment extraction function defined")

---
## 5. Answer Matching Algorithm

Matches each question from the question paper to its answer in the answer sheet.

### Scoring Formula:

```
score = 0

# Number matching (most important)
if question_number == segment_number:
    score += 55    # Exact match → strong positive
else:
    score -= 20    # Wrong number → penalty

# Letter matching (sub-questions)
if question_letter == segment_letter:
    score += 35    # Exact sub-question match
elif segment_letter is None:
    score += 5     # No letter specified (partial credit)
else:
    score -= 20    # Wrong letter → penalty

# Text similarity (weighted 35%)
similarity = partial_ratio * 0.45 + token_set_ratio * 0.40 + ratio * 0.15
score += similarity * 0.35
```

### Text Similarity (3 metrics combined):

| Metric | Weight | What it measures |
|--------|--------|------------------|
| `partial_ratio` | 45% | Best substring match |
| `token_set_ratio` | 40% | Word-level matching (ignores order) |
| `ratio` | 15% | Exact character-level match |

In [ ]:
def question_similarity(question_text, answer_text):
    """
    ALGORITHM: Fuzzy Text Similarity
    
    Uses rapidfuzz's 3 metrics:
    1. partial_ratio: Best matching substring (0-100)
       → "describe gravity" vs "gravity is a force"
       → partial: 100 ("gravity" found)
    
    2. token_set_ratio: Word-level set matching (0-100)
       → Same words regardless of order
       → {gravity, describe} vs {gravity, is, a, force}
       → token_set: 72
    
    3. ratio: Exact character match (0-100)
       → Only if strings are very similar
    
    Final score = partial * 0.45 + token * 0.40 + ratio * 0.15
    """
    q = normalize_text(question_text).lower()
    a = normalize_text(answer_text).lower()
    
    if not q or not a:
        return 0.0
    
    partial = fuzz.partial_ratio(q, a)
    token = fuzz.token_set_ratio(q, a)
    ratio = fuzz.ratio(q, a)
    
    return partial * 0.45 + token * 0.40 + ratio * 0.15

# Demo
q = "सिंधु घाटी सभ्यता की विशेषताएँ लिखिए"
a = "सिंधु घाटी सभ्यता शहरी योजना जल प्रबंधन"
print(f"Similarity: {question_similarity(q, a):.1f}")

q2 = "रोमन साम्राज्य और गुप्त साम्राज्य की तुलना"
a2 = "आज का मौसम अच्छा है"
print(f"Similarity (unrelated): {question_similarity(q2, a2):.1f}")

In [ ]:
def score_segment(question, segment):
    """
    ALGORITHM: Segment Scoring
    
    Combines number matching + letter matching + text similarity:
    
    Scoring breakdown:
    - Number match: +55 (or -20 penalty)
    - Letter match: +35 (or -20 penalty)
    - Text similarity * 0.35: 0-35 points
    
    Max possible score: ~125
    Min possible score: ~-40
    Threshold for match: 45
    """
    q_number, q_letter = parse_question_id(question.get("question_id"))
    s_number = segment["number"]
    s_letter = segment["letter"]
    
    score = 0.0
    
    # Number matching
    if q_number == s_number:
        score += 55
    else:
        score -= 20
    
    # Letter matching
    if q_letter:
        if s_letter == q_letter:
            score += 35
        elif s_letter is None:
            score += 5
        else:
            score -= 20
    else:
        if s_letter is None:
            score += 15
    
    # Text similarity
    similarity = question_similarity(
        question.get("question_text", ""),
        segment.get("raw_text", ""),
    )
    score += similarity * 0.35
    
    return score

print("✅ Segment scoring function defined")

In [ ]:
# Demo: Scoring example
question = {
    "question_id": "q_5",
    "question_text": "सिंधु घाटी सभ्यता की विशेषताएँ लिखिए"
}

correct_segment = {
    "number": 5, "letter": None,
    "raw_text": "5. सिंधु घाटी सभ्यता शहरी योजना जल प्रबंधन नगर नियोजन"
}

wrong_segment = {
    "number": 8, "letter": None,
    "raw_text": "8. रोमन साम्राज्य सीजर ऑगस्टस"
}

print(f"Correct segment score: {score_segment(question, correct_segment):.1f}")
print(f"Wrong segment score:   {score_segment(question, wrong_segment):.1f}")
print(f"\nThreshold for match: 45")
print(f"Correct > threshold: {score_segment(question, correct_segment) > 45}")
print(f"Wrong > threshold:   {score_segment(question, wrong_segment) > 45}")

---
## 6. OCR Highlighting Algorithm

Calculates bounding box coordinates for highlighting answers on the answer sheet.

### BBox Normalization:

1. Find the bounding box of all answer words
2. Convert pixel coordinates to **0-1000 scale** (per-mille)
3. Frontend renders highlight overlay using percentage positions

### Formula:
```
ymin = (min_y / page_height) * 1000
xmin = (min_x / page_width) * 1000
ymax = (max_y / page_height) * 1000
xmax = (max_x / page_width) * 1000
```

In [ ]:
def words_bbox(words, page_width, page_height):
    """
    ALGORITHM: Bounding Box Calculation
    
    Converts word coordinates to a normalized bounding box.
    
    1. Find min/max x and y across all words
    2. Convert to per-mille (0-1000) coordinates:
       ymin = (min_y / page_height) * 1000
       xmin = (min_x / page_width) * 1000
       ymax = (max_y / page_height) * 1000
       xmax = (max_x / page_width) * 1000
    
    Frontend rendering:
    - left:   (xmin / 1000) * 100%
    - top:    (ymin / 1000) * 100%
    - width:  ((xmax - xmin) / 1000) * 100%
    - height: ((ymax - ymin) / 1000) * 100%
    
    Returns: [ymin, xmin, ymax, xmax] in 0-1000 scale
    """
    if not words:
        return [0, 0, 0, 0]
    
    x0 = min(w["x0"] for w in words)
    y0 = min(w["y0"] for w in words)
    x1 = max(w["x1"] for w in words)
    y1 = max(w["y1"] for w in words)
    
    ymin = (y0 / page_height) * 1000
    xmin = (x0 / page_width) * 1000
    ymax = (y1 / page_height) * 1000
    xmax = (x1 / page_width) * 1000
    
    return [ymin, xmin, ymax, xmax]

# Demo
sample_words = [
    {"x0": 50, "y0": 100, "x1": 100, "y1": 120},
    {"x0": 110, "y0": 100, "x1": 160, "y1": 120},
    {"x0": 50, "y0": 125, "x1": 200, "y1": 145},
]
bbox = words_bbox(sample_words, page_width=600, page_height=800)
print(f"Bounding box: {bbox}")
print(f"Frontend position:")
print(f"  left:   {bbox[1]/10:.1f}%")
print(f"  top:    {bbox[0]/10:.1f}%")
print(f"  width:  {(bbox[3]-bbox[1])/10:.1f}%")
print(f"  height: {(bbox[2]-bbox[0])/10:.1f}%")

### 6b. Multi-page Answer Merging

When an answer spans multiple pages:

1. Find all segments with same question number
2. Sort by page number
3. First segment → main page + primary bounding box
4. Remaining segments → additional_pages array
5. Check markerless pages between marked pages → include as continuations

**Result**: `is_multipage: true` with `additional_pages` containing each page's bbox.

In [ ]:
def merge_multipage_segments(question, first_segment, all_segments, pages_with_markers):
    """
    ALGORITHM: Multi-page Answer Merging
    
    For answers that span multiple PDF pages:
    
    1. Collect all segments with same (number, letter)
    2. Include markerless pages between marked pages
    3. Sort by page number
    4. Build merged answer:
       - main: first page segment
       - additional_pages: remaining pages with individual bboxes
       - raw_text: all text concatenated
    
    Confidence: 0.92 for multi-page (high due to marker matching)
    """
    q_number, q_letter = parse_question_id(question.get("question_id"))
    if not first_segment:
        return None
    
    selected = [first_segment]
    first_page = first_segment["page_number"]
    
    # Find same-number segments on other pages
    for segment in all_segments:
        if segment is first_segment:
            continue
        if segment["number"] != q_number:
            continue
        if q_letter and segment["letter"] != q_letter:
            continue
        if segment["page_number"] == first_page:
            continue
        selected.append(segment)
    
    # Include markerless continuation pages
    for page_num in sorted(pages_with_markers.keys()):
        if page_num <= first_page:
            continue
        markers = pages_with_markers[page_num]
        has_marker = any(
            m["number"] == q_number and (q_letter is None or m["letter"] == q_letter)
            for m in markers
        )
        if not has_marker:
            cont = next((s for s in all_segments if s["page_number"] == page_num and s.get("is_continuation")), None)
            if cont:
                selected.append(cont)
    
    selected.sort(key=lambda x: x["page_number"])
    
    # Single page → simple answer
    if len(selected) == 1:
        return {
            "page_number": selected[0]["page_number"],
            "raw_text": selected[0]["raw_text"],
            "bounding_box": words_bbox(selected[0]["words"], selected[0]["page_width"], selected[0]["page_height"]),
            "is_multipage": False,
            "additional_pages": [],
            "confidence_score": 0.90,
        }
    
    # Multi-page answer
    main = selected[0]
    all_text = [s["raw_text"] for s in selected]
    
    additional_pages = []
    for seg in selected[1:]:
        additional_pages.append({
            "page_number": seg["page_number"],
            "bounding_box": words_bbox(seg["words"], seg["page_width"], seg["page_height"]),
            "raw_text": seg["raw_text"],
        })
    
    return {
        "page_number": main["page_number"],
        "raw_text": normalize_text(" ".join(all_text)),
        "bounding_box": words_bbox(main["words"], main["page_width"], main["page_height"]),
        "is_multipage": True,
        "additional_pages": additional_pages,
        "confidence_score": 0.92,
    }

print("✅ Multi-page merging function defined")

---
## 7. AI Evaluation / Grading Algorithm

Uses **Gemini API** with subject-specific grading rubrics.

### Subject Detection:

Question keywords → Subject → Grading rubric:

| Keywords | Subject | Grading Focus |
|----------|---------|---------------|
| newton, force, energy, velocity | Physics | Formulas, units, calculations |
| chemical, reaction, element, pH | Chemistry | Equations, balancing, reactions |
| synonym, passive, tense | English | Grammar, vocabulary, structure |
| विवेचना, सभ्यता, इतिहास, गांधी | History | Facts, dates, analysis |
| (default) | Generic | Content completeness, clarity |

In [ ]:
def detect_subject(question_text):
    """
    ALGORITHM: Subject Detection via Keyword Matching
    
    Maps question keywords to academic subjects.
    Each subject has its own grading rubric.
    
    Detection is done by checking if any keyword appears
    in the lowercased question text.
    """
    q = question_text.lower()
    
    physics_keywords = [
        "newton", "force", "acceleration", "speed", "velocity",
        "power", "energy", "lift", "distance", "physics", "mass",
    ]
    
    chemistry_keywords = [
        "chemical", "reaction", "element", "compound", "mixture",
        "isotopes", "ph", "balanced", "chemistry", "acid", "base",
    ]
    
    english_keywords = [
        "synonym", "passive", "tense", "preposition",
        "figure of speech", "paragraph", "english", "grammar",
    ]
    
    history_keywords = [
        "विवेचना", "वर्णन", "सभ्यता", "साम्राज्य", "क्रांति",
        "आंदोलन", "पतन", "कारण", "प्रभाव", "तुलना", "विशेषता",
        "इतिहास", "भारत", "गांधी", "स्वतंत्रता", "मुगल", "गुप्त",
    ]
    
    if any(k in q for k in physics_keywords):
        return "physics"
    if any(k in q for k in chemistry_keywords):
        return "chemistry"
    if any(k in q for k in english_keywords):
        return "english"
    if any(k in q for k in history_keywords):
        return "history"
    
    return "generic"

# Demo
tests = [
    "न्यूटन के गति के नियम लिखिए",
    "रासायनिक अभिक्रिया के प्रकार लिखिए",
    "सिंधु घाटी सभ्यता की विशेषताएँ",
    "What is a synonym?",
    "बताइए कि पृथ्वी क्यों घूमती है",
]

for t in tests:
    print(f"  '{t[:30]}...' → {detect_subject(t)}")

### 7b. Gemini AI Grading Prompt

Each subject has a **custom grading rubric** sent to Gemini:

```python
# Example: History grading prompt
prompt = f"""
You are VedaAI, an expert Indian history examiner.

QUESTION:
{question}

STUDENT ANSWER:
{student_answer}

Evaluate using HISTORY GRADING RULES:

1. विवेचना / वर्णन type:
   - Must explain with 3-4 clear points
   - Must include specific facts, dates, names
   - Missing points = proportionally reduced score

2. तुलना type:
   - Must mention BOTH items
   - Must give specific differences

3. कारण type:
   - Must explain cause AND effect
   - Missing either = max 5/10

Return JSON: {score, feedback, strengths[], improvements[]}
"""
```

In [ ]:
# Grading rubric summary for each subject
GRADING_RUBRICS = {
    "physics": {
        "focus": ["Formulas", "Units", "Calculations", "Diagrams"],
        "deductions": {
            "missing_formula": -6,
            "missing_units": -2,
            "wrong_calculation": -4,
            "missing_steps": -2,
        },
        "scoring": "9-10: Complete+accurate | 7-8: Minor omission | 5-6: Partially correct | 3-4: Incomplete | 1-2: Wrong"
    },
    "chemistry": {
        "focus": ["Equations", "Balancing", "Reactions", "Properties"],
        "deductions": {
            "unbalanced_equation": -4,
            "missing_products": -3,
            "wrong_state": -1,
        },
        "scoring": "9-10: Balanced+complete | 7-8: Minor error | 5-6: Partially balanced | 3-4: Wrong approach | 1-2: Incorrect"
    },
    "history": {
        "focus": ["Dates", "Events", "Analysis", "Comparisons"],
        "deductions": {
            "no_dates": -3,
            "no_analysis": -4,
            "vague_answer": -5,
        },
        "scoring": "9-10: Complete+analytical | 7-8: Good coverage | 5-6: Basic facts only | 3-4: Incomplete | 1-2: Irrelevant"
    },
    "english": {
        "focus": ["Grammar", "Vocabulary", "Structure", "Examples"],
        "deductions": {
            "grammar_errors": -2,
            "no_examples": -3,
            "poor_structure": -2,
        },
        "scoring": "9-10: Fluent+accurate | 7-8: Minor errors | 5-6: Understandable | 3-4: Poor grammar | 1-2: Incomprehensible"
    },
    "generic": {
        "focus": ["Content", "Clarity", "Relevance", "Completeness"],
        "deductions": {},
        "scoring": "9-10: Excellent | 7-8: Good | 5-6: Average | 3-4: Below avg | 1-2: Poor"
    },
}

for subject, rubric in GRADING_RUBRICS.items():
    print(f"\n{subject.upper()}:")
    print(f"  Focus: {rubric['focus']}")
    print(f"  Scoring: {rubric['scoring']}")

### 7c. Local Fallback Grading

When Gemini API is unavailable, the **local fallback** generates answer-specific feedback:

1. Extract topic keywords from question (nouns, proper nouns)
2. Count answer length in words
3. Check if keywords from question appear in answer
4. Score based on:
   - Coverage: how many question keywords are in the answer
   - Length: word count thresholds
   - Relevance: keyword match ratio
5. Generate unique feedback referencing actual answer content

In [ ]:
def extract_keywords(text):
    """
    ALGORITHM: Keyword Extraction for Grading
    
    1. Remove stop words and common words
    2. Extract words >= 3 characters
    3. Return unique keywords (preserving order)
    """
    STOP_WORDS = {
        "the", "is", "in", "of", "and", "to", "a", "an", "for",
        "that", "it", "on", "with", "as", "at", "by", "from",
        "लिखिए", "बताइए", "कीजिए", "करें", "है", "के", "को",
        "में", "से", "और", "एक", "यह", "वह", "पर", "ने",
    }
    words = re.findall(r"[\w\u0900-\u097F]{3,}", text.lower())
    return [w for w in words if w not in STOP_WORDS]

def local_grade(question, answer):
    """
    ALGORITHM: Local Fallback Grading
    
    No Gemini API needed — pure keyword analysis:
    
    1. Extract keywords from question
    2. Check keyword coverage in answer
    3. Analyze answer length
    4. Generate topic-specific feedback
    5. Score: 0-10 based on coverage + length
    """
    q_keywords = extract_keywords(question)
    a_words = answer.split()
    a_length = len(a_words)
    
    # Keyword coverage
    matched = [k for k in q_keywords if k in answer.lower()]
    coverage = len(matched) / max(len(q_keywords), 1)
    
    # Length scoring
    if a_length < 5:
        length_score = 1
    elif a_length < 15:
        length_score = 4
    elif a_length < 30:
        length_score = 7
    else:
        length_score = 9
    
    # Combined score
    score = round(coverage * 5 + length_score * 0.5)
    score = max(0, min(10, score))
    
    # Generate feedback
    missing = [k for k in q_keywords if k not in answer.lower()]
    missing_str = ", ".join(missing[:3]) if missing else "none"
    
    feedback = f"Answer has {a_length} words with {coverage*100:.0f}% keyword coverage. "
    feedback += f"Missing concepts: {missing_str}. "
    feedback += f"Current content: {' '.join(a_words[:5])}..."
    
    strengths = []
    improvements = []
    
    if coverage > 0.5:
        strengths.append(f"Covers {len(matched)} key concepts from the question")
    if a_length > 20:
        strengths.append("Provides detailed explanation")
    if missing:
        improvements.append(f"Include: {', '.join(missing[:3])}")
    if a_length < 10:
        improvements.append("Answer is too brief — expand with more details")
    
    return {
        "score": score,
        "feedback": feedback,
        "strengths": strengths,
        "improvements": improvements,
    }

# Demo
result = local_grade(
    "सिंधु घाटी सभ्यता की विशेषताएँ लिखिए",
    "सिंधु घाटी में शहरी योजना थी जल प्रबंधन था नगर नियोजन था"
)
print(json.dumps(result, indent=2, ensure_ascii=False))

---
## 8. Deduplication Algorithm

Removes duplicate answer segments that might be detected multiple times:

1. **Key**: `(question_number, letter, page_number)`
2. If same key appears twice → keep the one with longer text
3. This handles cases where OCR detects the same marker twice

In [ ]:
def deduplicate_segments(segments):
    """
    ALGORITHM: Segment Deduplication
    
    When OCR detects the same question marker multiple times
    (e.g., from different OCR passes), keep the best version.
    
    Key: (number, letter, page_number)
    Winner: Segment with longest raw_text
    """
    seen = {}
    deduped = []
    
    for segment in segments:
        key = (segment["number"], segment["letter"], segment["page_number"])
        
        if key not in seen:
            seen[key] = segment
            deduped.append(segment)
        else:
            # Keep the longer text
            existing = seen[key]
            if len(segment.get("raw_text", "")) > len(existing.get("raw_text", "")):
                seen[key] = segment
                deduped = [s for s in deduped if not (
                    s["number"] == key[0] and s["letter"] == key[1] and s["page_number"] == key[2]
                )]
                deduped.append(segment)
    
    return deduped

# Demo
segments = [
    {"number": 5, "letter": None, "page_number": 1, "raw_text": "5. short"},
    {"number": 5, "letter": None, "page_number": 1, "raw_text": "5. सिंधु घाटी सभ्यता शहरी योजना जल प्रबंधन"},
    {"number": 8, "letter": None, "page_number": 1, "raw_text": "8. रोमन साम्राज्य"},
]

result = deduplicate_segments(segments)
print(f"Before: {len(segments)} segments")
print(f"After:  {len(result)} segments")
for s in result:
    print(f"  Q{s['number']}: '{s['raw_text'][:40]}...'")

---
## 9. Complete Pipeline Summary

```
INPUT: Question Paper (PDF/Image) + Answer Sheet (PDF/Image)
│
├─ 1. TEXT EXTRACTION
│   ├── PDF → PyMuPDF (or Tesseract fallback)
│   └── Image → Gemini Vision OCR
│
├─ 2. QUESTION PARSING
│   ├── Regex: Q/प्रश्न/Question + number
│   ├── Extract text block until next question
│   └── Detect sub-questions (a), (b), (c)
│
├─ 3. ANSWER SEGMENT DETECTION
│   ├── PASS 1: Multi-word markers ("Answer for Question N")
│   ├── PASS 2: Single-word markers ("5.", "Q5")
│   ├── PASS 3: Split markers ("Q" + "5" + "(a)")
│   └── Extract segments between markers
│
├─ 4. ANSWER MATCHING
│   ├── Number matching (+55 / -20)
│   ├── Letter matching (+35 / -20)
│   ├── Fuzzy text similarity * 0.35
│   └── Threshold: score > 45
│
├─ 5. MULTI-PAGE MERGING
│   ├── Collect same-number segments across pages
│   ├── Include markerless continuation pages
│   └── Build per-page bounding boxes
│
├─ 6. OCR HIGHLIGHTING
│   ├── Calculate bbox from word coordinates
│   ├── Normalize to 0-1000 scale
│   └── Frontend renders highlight overlay
│
├─ 7. AI EVALUATION
│   ├── Subject detection (physics/chemistry/history/english)
│   ├── Gemini API with subject-specific rubric
│   ├── Local fallback: keyword analysis
│   └── Output: score, feedback, strengths, improvements
│
└─ 8. RESULT
    ├── Questions with scores and feedback
    ├── Answer sheet with highlighted regions
    └── Unmapped segments (if any)
```

### Performance Metrics:

| Stage | Time (PDF) | Time (Image) |
|-------|-----------|-------------|
| Text extraction | ~0.5s | ~2-3s (Gemini) |
| Question parsing | ~0.1s | ~0.1s |
| Answer segmentation | ~0.3s | ~0.3s |
| Answer matching | ~0.2s | ~0.2s |
| AI evaluation | ~3-5s (Gemini) | ~3-5s (Gemini) |
| **Total** | **~4-6s** | **~6-9s** |

In [ ]:
print("""
╔══════════════════════════════════════════════════╗
║       VedaAI Algorithm Documentation            ║
║       Complete Pipeline Reference               ║
╠══════════════════════════════════════════════════╣
║                                                  ║
║  8 algorithms documented:                        ║
║  1. Text Extraction (PyMuPDF/Tesseract/Gemini)   ║
║  2. Text Normalization                           ║
║  3. Question Parsing (Regex)                     ║
║  4. Answer Segment Detection (Multi-pass)        ║
║  5. Answer Matching (Fuzzy scoring)              ║
║  6. OCR Highlighting (BBox normalization)        ║
║  7. AI Evaluation (Subject-aware grading)        ║
║  8. Deduplication                                ║
║                                                  ║
╚══════════════════════════════════════════════════╝
""")